In [59]:
# 创建MongoDB客户端连接

import pymongo
client = pymongo.MongoClient("mongodb://localhost:27017/")

# 测试连接
try:
    client.admin.command('ping')
    print("MongoDB连接成功！")
except Exception as e:
    print(f"MongoDB连接失败：{e}")

MongoDB连接成功！


In [60]:
from dotenv import load_dotenv
load_dotenv()  # 必须要这句！
import os
from typing import Literal

from langchain_core.tools import tool
# 替换为 DashScope (通义千问) 的集成类
from langchain_community.chat_models.tongyi import ChatTongyi
from langgraph.prebuilt import create_react_agent

@tool
def get_weather(city: Literal["北京", "深圳"]):
    """用来返回天气信息的工具函数。"""
    if city == "北京":
        return "北京天气晴朗 大约22度 湿度30%"
    elif city == "深圳":
        return "深圳天气多云 大约28度 湿度80%"
    else:
        raise AssertionError("Unknown city")

tools = [get_weather]

# 修改为 Qwen-Plus 的配置
model = ChatTongyi(
    model="qwen-plus",
    temperature=0,
    # 确保你设置了环境变量 DASHSCOPE_API_KEY
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

# 后续可以继续创建 agent
# agent_executor = create_react_agent(model, tools)

In [61]:
from langgraph.checkpoint.mongodb import MongoDBSaver
from langgraph.prebuilt import create_react_agent
MONGODB_URI = "mongodb://localhost:27017"  # 替换为你的实际连接字符串

# 3. 使用 MongoDB 作为 Checkpointer 创建并运行 Agent
with MongoDBSaver.from_conn_string(MONGODB_URI) as checkpointer:
    # 将适配好的 qwen 模型传入
    graph = create_react_agent(model, tools=tools, checkpointer=checkpointer)

    # 配置线程 ID，用于识别特定的对话会话
    config = {"configurable": {"thread_id": "1"}}

    # 调用 Agent
    response = graph.invoke(
        {"messages": [("human", "北京今天的天气如何？")]},
        config
    )

# 打印结果
print(response["messages"][-1].content)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_27584\816267848.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = create_react_agent(model, tools=tools, checkpointer=checkpointer)


北京今天的天气晴朗，气温约为22摄氏度，湿度为30%。


进行消息过滤和总结上下文，防止撑爆

In [62]:
from langchain_openai import ChatOpenAI
from typing import Literal

from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import ToolNode
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
load_dotenv()  # 必须要这句！
api_key = os.environ.get("DASHSCOPE_API_KEY")
base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
memory = MemorySaver()
@tool
def search(query:str):
    """调用此模型可以浏览网络"""
    return "北京天气晴朗，气温约为22摄氏度，湿度为30%"
tools = [search]
tool_node = ToolNode(tools)
model = ChatOpenAI(
    model="qwen-plus",
    temperature=0,
    api_key=api_key,
    base_url=base_url
)
bound_model=model.bind_tools(tools)
def should_continue(state: MessagesState):
    """返回下一个要执行的节点。"""
    last_message = state["messages"][-1]
    # 如果没有函数调用，则结束
    if not last_message.tool_calls:
        return END
    # 否则，如果有函数调用，我们继续
    return "action"


def filter_messages(messages: list):
    # 这是一个非常简单的辅助函数，它只使用最后一条消息
    return messages[-1:]

# 定义调用模型的函数

def call_model(state: MessagesState):
    # 1. 依然保留你的过滤逻辑
    messages = filter_messages(state["messages"])

    # 2. 获取存储在 State 中的摘要
    summary = state.get("summary", "")

    # 3. 如果有摘要，将其作为 SystemMessage 插入到消息最前面
    if summary:
        from langchain_core.messages import SystemMessage
        system_message = SystemMessage(content=f"这是之前的对话摘要，请参考：{summary}")
        messages = [system_message] + messages

    # 4. 调用模型
    response = bound_model.invoke(messages)
    return {"messages": response}



In [63]:
from langchain_core.messages import RemoveMessage
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# 定义状态结构
class State(TypedDict):
    # 使用 Annotated 确保消息是追加/更新，而不是覆盖
    messages: Annotated[list, add_messages]
    # 用于存储对话摘要的字符串
    summary: str
def summarize_conversation(state: State):
    # 首先，我们获取现有的摘要
    summary = state.get("summary", "")

    if summary:
        # 如果已经存在摘要，我们要求模型根据新消息扩展它
        summary_message = (
            f"这是迄今为止对话的摘要：{summary}\n\n"
            "考虑上面的新消息，扩展摘要："
        )
    else:
        # 如果没有摘要，则请求创建第一个摘要
        summary_message = "创建上述对话的摘要："

    # 将摘要请求作为一条新消息发给模型
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)

    # --- 核心清理逻辑 ---
    # 我们生成 RemoveMessage 指令，删除除了最后两条以外的所有历史消息
    # 这样可以保持 State 极其精简，只留下摘要和最新的互动
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    print(f"DEBUG: 当前读取到的摘要内容是: {summary}")
    return {
        "summary": response.content,
        "messages": delete_messages
    }


In [64]:
# 1. 定义图
workflow = StateGraph(State) # 注意这里改用你定义的那个包含 summary 字段的 State

# 2. 添加所有节点
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)
workflow.add_node("summarize", summarize_conversation) # 添加总结节点

# 3. 设置逻辑连线
workflow.add_edge(START, "agent")

# 4. 修改条件边逻辑
workflow.add_conditional_edges(
    "agent",
    should_continue, #
    {
        "action": "action",
        # 如果模型不需要调用工具了，就去进行“总结”
        END: "summarize"
    },
)

# 5. 工具执行完回 agent
workflow.add_edge("action", "agent")

# 6. 总结完后正式结束
workflow.add_edge("summarize", END)

# 7. 编译
app = workflow.compile(checkpointer=memory)

由于使用了消息裁剪 这里是记不住对话的

In [65]:
from langchain_core.messages import HumanMessage

# 配置线程 ID，用于区分不同的用户或会话
config = {"configurable": {"thread_id": "2"}}

# 第一轮对话：介绍自己
input_message = HumanMessage(content="hi! 我是tomie")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

# ---------------------------------------------------------
# 请注意，我们在这里使用了一个辅助函数，它只使用最后一条消息
# 这将导致我们的模型只看到最后一条消息
# ---------------------------------------------------------

# 第二轮对话：测试记忆
input_message = HumanMessage(content="我叫什么名字?")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

hi! 我是tomie
================================== Ai Message ==================================

Hi Tomie! 很高兴认识你！😊 有什么我可以帮你的吗？
DEBUG: 当前读取到的摘要内容是: 
================================== Ai Message ==================================

Hi Tomie! 很高兴认识你！😊 有什么我可以帮你的吗？
================================ Human Message =================================

我叫什么名字?
================================== Ai Message ==================================

我无法知道您的名字，因为您没有告诉我。如果您愿意，可以告诉我您的名字！
DEBUG: 当前读取到的摘要内容是: 对话摘要：  
用户Tomie向助手打招呼并自我介绍，助手热情回应，表示很高兴认识Tomie，并主动询问是否需要帮助。
================================== Ai Message ==================================

我无法知道您的名字，因为您没有告诉我。如果您愿意，可以告诉我您的名字！


使用对话总结技术 升级一下print 函数 看看记忆过程

In [66]:
def print_update(update):
    for node_name, v in update.items():
        # 获取 messages 键对应的内容
        msgs_data = v.get("messages")

        if msgs_data:
            # 核心兼容逻辑：如果 msgs_data 是单个对象，就包装成列表；如果是列表就直接用
            messages_list = msgs_data if isinstance(msgs_data, list) else [msgs_data]

            for m in messages_list:
                # 自动解包元组 (BaseMessage, metadata)
                msg = m[0] if isinstance(m, tuple) else m

                # 获取内容和类型
                content = getattr(msg, "content", "")
                msg_type = getattr(msg, "type", "")

                if content and str(content).strip():
                    if msg_type == "ai":
                        print(f"【AI】: {content}")
                    elif msg_type == "human":
                        print(f"【用户】: {content}")
                    else:
                        print(f"【{msg_type}】: {content}")

        if "summary" in v:
            print(f"--- 摘要更新 ---: {v['summary']}")

In [67]:
# 第一轮
input_message = HumanMessage(content="hi! 我是tomie")
# 删掉这一行：input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

# 第二轮
input_message = HumanMessage(content="我叫什么名字?")
# 删掉这一行：input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

# 第三轮
input_message = HumanMessage(content="我喜欢AI应用开发!")
# 删掉这一行：input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

【AI】: Hi Tomie! 很高兴认识你！😊 有什么我可以帮你的吗？
DEBUG: 当前读取到的摘要内容是: 对话摘要（更新版）：  
用户Tomie首次打招呼并自我介绍：“hi! 我是tomie”。助手热情回应，表示很高兴认识Tomie，并主动提供帮助。随后，Tomie再次确认自己的名字：“我叫什么名字？”，助手诚恳说明自己无法预先知晓用户姓名，但欢迎Tomie主动告知——体现了尊重隐私、注重互动真实性的沟通原则。整个对话简洁友好，建立了初步的信任与亲切感。
--- 摘要更新 ---: 更新后的对话摘要（截至当前轮次）：  
用户Tomie以轻松友好的方式开启对话：“hi! 我是tomie”，完成了自我介绍；助手随即给予温暖、个性化的回应——直呼其名（“Hi Tomie!”），表达喜悦，并主动开放支持意愿，强化了人际联结感。随后，Tomie特意回溯提问“我叫什么名字？”，既可能出于测试、确认，也可能是 playful 的互动尝试；助手未回避或假设，而是坦诚说明AI无预设信息、尊重用户自主披露权，同时保持开放与谦和态度，体现了隐私意识、诚实原则与以用户为中心的服务理念。两轮交互虽简短，但已自然建立起清晰的身份锚点（Tomie）、积极的情绪基调（友好、信任）和健康的沟通范式（真诚、不越界、有边界感）。对话目前处于温暖启程阶段，为后续个性化交流奠定了良好基础。
【AI】: 我无法知道您的名字，因为您没有告诉我。如果您愿意分享，我很乐意知道！
DEBUG: 当前读取到的摘要内容是: 更新后的对话摘要（截至当前轮次）：  
用户Tomie以轻松友好的方式开启对话：“hi! 我是tomie”，完成了自我介绍；助手随即给予温暖、个性化的回应——直呼其名（“Hi Tomie!”），表达喜悦，并主动开放支持意愿，强化了人际联结感。随后，Tomie特意回溯提问“我叫什么名字？”，既可能出于测试、确认，也可能是 playful 的互动尝试；助手未回避或假设，而是坦诚说明AI无预设信息、尊重用户自主披露权，同时保持开放与谦和态度，体现了隐私意识、诚实原则与以用户为中心的服务理念。两轮交互虽简短，但已自然建立起清晰的身份锚点（Tomie）、积极的情绪基调（友好、信任）和健康的沟通范式（真诚、不越界、有边界感）。对话目前处于温暖启程阶段，为后续个性化交流奠定了良好基础。
--- 摘要更新